In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow import keras
from keras.datasets import fashion_mnist
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)

## 1) Förbereda datan

- ladda in datan
- undersöka vilka klasser som finns
- se hur många bilder som finns
- identifiera eventuella obalanser


In [ ]:
#Ladda in datan
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images_full, train_labels_full), (test_images, test_labels) = fashion_mnist.load_data()


print("Bilder i train set:", train_images_full.shape)
print("Bilder i test set:", test_images.shape)
print("Bilder totalt:", len(train_labels_full) + len(test_labels))
print("Klasser:", np.unique(test_labels))

### Summering av dataset

- Datasetet innehåller 70,000 bilder där varje bild representeras i 28 x 28 pixels var.
- Datasetet är redan indelat i train-set - 60,000 bilder och test-set - 10,000. 
- Det finns 10 klasskategorier eller *labels* som bilderna kan hamna inom - *labels* är en vektor av integers från 0 till 9. Dessa representerar vilken sorts klädkategori bilder hamnar inom:

| Label       | Class       |
| ----------- | ----------- |
| 0 	      | T-shirt/top |
| 1 	      |  Trouser    |
| 2 	      |  Pullover   |
| 3 	      |  Dress      |
| 4 	      |  Coat       |
| 5 	      |  Sandal     |
| 6 	      |  Shirt      |
| 7 	      |  Sneaker    |
| 8 	      |  Bag        |
| 9 	      |  Ankle boot |

In [ ]:
# Identifiera eventuella obalanser

classes_train, counts_train = np.unique(train_labels_full, return_counts=True)
classes_test, counts_test = np.unique(test_labels, return_counts=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(classes_train, counts_train, color = 'blue')
axes[0].set_title('Klassfördelning i train set')
axes[0].set_xlabel('Klass')
axes[0].set_ylabel('Antal')

axes[1].bar(classes_test, counts_test, color = 'green')
axes[1].set_title('Klassfördelning i test set')
axes[1].set_xlabel('Klass')
axes[1].set_ylabel('Antal')

plt.tight_layout()
plt.show()


### Summering av klassfördelning

Klassfördelningen visar ett perfekt balanserat dataset igenom både train och test setet. 

In [ ]:
# Bild före preprocessing

plt.figure()
plt.imshow(train_images_full[0])
plt.colorbar()
plt.grid(False)
plt.show()

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Testa att normaliseringen funkade
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images_full[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels_full[i]])
plt.show()


## 2) Förbereda datan

- forma datan så att den fungerar med modellen:
    - skapa ett subset av träningsdatan
    - dela datan i train - val 
    - normalisera/skala bilderna


In [ ]:
def create_balanced_subset(X, y, samples_per_class=700, random_state=42):

    rng = np.random.default_rng(random_state)
    selected_indices = []

    for class_id in np.unique(y):
        class_indices = np.where(y == class_id)[0]

        if len(class_indices) < samples_per_class:
            raise ValueError(
                f"Klassen {class_id} har bara {len(class_indices)} exempel, "
                f"men samples_per_class={samples_per_class}"
            )
        
        chosen_indices = rng.choice(
            class_indices,
            size=samples_per_class,
            replace=False
        )

        selected_indices.extend(chosen_indices)
    
    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)

    return X[selected_indices], y[selected_indices]


X_subset_raw, y_subset = create_balanced_subset(
    train_images_full,
    train_labels_full,
    samples_per_class=700,
    random_state=42
)

print("X_subset_raw:", X_subset_raw.shape)
print("y_subset:", y_subset.shape)



In [ ]:
train_images, val_images, train_labels, val_label = train_test_split(
    X_subset_raw,
    y_subset,
    test_size=0.20,
    random_state=42,
    stratify=y_subset
)

print("Train images:", train_images.shape)
print("Validation images:", val_images.shape)
print("Train labels:", train_labels.shape)
print("Validatio labels:", val_label.shape)

In [ ]:
train_images_norm = train_images.astype("float32") / 255.0
val_images_norm = val_images.astype("float32") / 255.0
test_images_norm = test_images.astype("float32") / 255.0

print("Train imgs min/max:", train_images_norm.min(), train_images_norm.max())
print("Validation imgs min/max:", val_images_norm.min(), val_images_norm.max())
print("Train imgs shape:", test_images_norm.shape)

In [ ]:
train_images_norm = train_images_norm[..., np.newaxis]
val_images_norm = val_images_norm[..., np.newaxis]
test_images_norm = test_images_norm[..., np.newaxis]

print("Train imgs shape efter kanal-dimension:", train_images_norm.shape)
print("Validation imgs shape efter kanal-dimension:", val_images_norm.shape)
print("Test imgs shape efter kanal-dimension:", test_images_norm.shape)